In [11]:
import os
from pathlib import Path

import numpy as np
import nibabel as nib

# Rutas de los datos
BASE_DIR = Path("/users/lvelez/PI_Velez/data/Couinaud")
TR_DIR   = BASE_DIR / "CouinaudTr"
TS_DIR   = BASE_DIR / "CouinaudTs"

# Parámetros de ventaneo
WIN_MIN = -200.0
WIN_MAX =  250.0
DENOM   = WIN_MAX - WIN_MIN  # 450

In [12]:
def window_and_normalize(img_data, wmin=WIN_MIN, wmax=WIN_MAX):
    """
    Aplica ventaneo [wmin, wmax] y normaliza a [0,1].
    img_data: numpy array con intensidades (HU).
    """
    # Ventaneo: recortar intensidades fuera de [wmin, wmax]
    img = np.clip(img_data, wmin, wmax)

    # Normalizar a [0,1]
    img = (img - wmin) / (wmax - wmin)

    # Por seguridad, recortar de nuevo a [0,1]
    img = np.clip(img, 0.0, 1.0)

    # Usamos float32 para ahorrar algo de espacio
    return img.astype(np.float32)

In [13]:
def process_folder(folder_path):
    """
    Recorre una carpeta, toma todos los .nii.gz que NO tengan '_ventaneo'
    en el nombre, les aplica ventaneo+normalización y guarda un nuevo archivo
    con sufijo '_ventaneo.nii.gz' en la misma carpeta.
    """
    folder_path = Path(folder_path)
    print(f"\nProcesando carpeta: {folder_path}")

    nii_files = sorted(folder_path.glob("*.nii.gz"))
    if not nii_files:
        print("  (No se encontraron .nii.gz)")
        return

    for nii_path in nii_files:
        # Saltar los que ya fueron procesados
        if "_ventaneo" in nii_path.name:
            print(f"  Skip (ya ventaneado): {nii_path.name}")
            continue

        print(f"  Leyendo: {nii_path.name}")
        img = nib.load(str(nii_path))
        data = img.get_fdata()  # numpy array

        # Aplicar ventaneo + normalización
        data_wn = window_and_normalize(data)

        # Crear nuevo nombre con sufijo
        new_name = nii_path.stem  # ej: "hepaticvessel_001" de "hepaticvessel_001.nii.gz"
        # ojo: stem quita SOLO ".gz"; si quieres quitar ".nii.gz" mejor:
        if new_name.endswith(".nii"):
            new_name = new_name[:-4]

        new_name = new_name + "_ventaneo.nii.gz"
        out_path = nii_path.parent / new_name

        # Crear nueva imagen NIfTI (manteniendo affine y header)
        new_img = nib.Nifti1Image(data_wn, affine=img.affine, header=img.header)
        # Forzamos el dtype del header a float32 para que no intente castear a otro tipo
        new_img.set_data_dtype(np.float32)

        nib.save(new_img, str(out_path))
        print(f"  Guardado: {out_path.name}")

    print("  ✔ Carpeta procesada.")

In [ ]:
# Procesar las dos carpetas
process_folder(TR_DIR)
process_folder(TS_DIR)

print("\n>>> Ventaneo y normalización completados para CouinaudTr y CouinaudTs.")

In [14]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

def list_pairs(folder_path):
    """
    Devuelve una lista de pares (orig_path, vent_path) en la carpeta.
    Considera como ventaneadas las que terminan en '_ventaneo.nii.gz'.
    """
    folder_path = Path(folder_path)
    vent_files = sorted(folder_path.glob("*_ventaneo.nii.gz"))
    pairs = []

    for vent_path in vent_files:
        # nombre original sin el sufijo '_ventaneo'
        base_name = vent_path.name.replace("_ventaneo", "")
        orig_path = folder_path / base_name
        if orig_path.exists():
            pairs.append((orig_path, vent_path))
        else:
            print(f"[Aviso] No encontré original para: {vent_path.name}")

    return pairs


def load_volume_3d(nii_path):
    """
    Carga un .nii.gz y devuelve un volumen numpy [Z, H, W]
    (reordenando ejes si es necesario).
    """
    img = nib.load(str(nii_path))
    data = img.get_fdata()  # típicamente [H, W, Z] o [X, Y, Z]

    # Aseguramos un array 3D
    if data.ndim != 3:
        raise ValueError(f"{nii_path} tiene {data.ndim} dimensiones, esperaba 3.")

    # Reordenamos a [Z, H, W] moviendo el último eje al primero
    # (si ya estuviera en [Z,H,W], esto igualmente funciona)
    vol = np.moveaxis(data, -1, 0)  # ahora vol.shape = [Z, H, W]
    return vol


def browse_original_vs_ventaneo(folder="train", pair_idx=0, k_rot=1):
    """
    Visualiza un paciente (volumen) con:
      - Izquierda: original
      - Derecha: ventaneado
    con un slider para recorrer slices.

    folder: 'train' -> CouinaudTr, 'test' -> CouinaudTs
    pair_idx: índice del par en esa carpeta
    k_rot: número de rotaciones de 90° (0,1,2,3) para ajustar orientación.
    """
    if folder == "train":
        root = TR_DIR
    else:
        root = TS_DIR

    pairs = list_pairs(root)
    if not pairs:
        print(f"No encontré pares original/ventaneo en {root}")
        return

    if pair_idx < 0 or pair_idx >= len(pairs):
        print(f"pair_idx fuera de rango. Hay {len(pairs)} pares.")
        return

    orig_path, vent_path = pairs[pair_idx]
    print(f"Carpeta : {root}")
    print(f"Par idx : {pair_idx}")
    print(f"Original: {orig_path.name}")
    print(f"Ventaneo: {vent_path.name}")

    vol_orig = load_volume_3d(orig_path)   # [Z, H, W]
    vol_vent = load_volume_3d(vent_path)  # [Z, H, W]

    if vol_orig.shape != vol_vent.shape:
        print("Ojo: original y ventaneado NO tienen la misma forma:",
              vol_orig.shape, vol_vent.shape)

    num_slices = vol_orig.shape[0]

    def show_slice(z):
        slice_orig = np.rot90(vol_orig[z], k=k_rot)
        slice_vent = np.rot90(vol_vent[z], k=k_rot)

        fig, axes = plt.subplots(1, 2, figsize=(10, 4))

        axes[0].imshow(slice_orig, cmap="gray")
        axes[0].set_title(f"Original (slice {z})")
        axes[0].axis("off")

        im = axes[1].imshow(slice_vent, cmap="gray")
        axes[1].set_title("Ventaneado")
        axes[1].axis("off")
        fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

        plt.tight_layout()
        plt.show()

    interact(
        show_slice,
        z=IntSlider(min=0, max=num_slices - 1, step=1, value=num_slices // 2)
    )


# Widget para elegir carpeta (train/test) y par
def browse_widget():
    folder_dd = Dropdown(options=[("Train (CouinaudTr)", "train"),
                                  ("Test (CouinaudTs)", "test")],
                         value="train",
                         description="Folder:")

    def inner(folder, pair_idx, k_rot):
        browse_original_vs_ventaneo(folder=folder, pair_idx=pair_idx, k_rot=k_rot)

    # Por defecto usamos pair_idx=0 y k_rot=1
    interact(
        inner,
        folder=folder_dd,
        pair_idx=IntSlider(min=0, max=20, step=1, value=0, description="Par idx"),
        k_rot=IntSlider(min=0, max=3, step=1, value=1, description="Rot k"),
    )

In [ ]:
browse_widget()

In [21]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import pandas as pd
from matplotlib.colors import ListedColormap, BoundaryNorm

def explorar_mascara_desde_archivos(folder="train", mask_idx=0, k_rot=1):
    """
    Visualiza una máscara de segmentación Couinaud directamente desde los .nii,
    usando un color fijo por clase (0..8).
    """
    # Elegir carpeta
    if folder == "train":
        root = TR_DIR
    else:
        root = TS_DIR

    masks = list_masks(root)
    if not masks:
        print(f"No encontré máscaras .nii en {root}")
        return

    if mask_idx < 0 or mask_idx >= len(masks):
        print(f"mask_idx fuera de rango. Hay {len(masks)} máscaras en {root}.")
        return

    # Mostrar algunas máscaras para que veas el índice
    print(f"\nCarpeta: {root}")
    print(f"Total de máscaras encontradas: {len(masks)}")
    print("Primeros 10 archivos de máscara (con su índice):")
    for i, p in enumerate(masks[:10]):
        print(f"  [{i}] {p.name}")
    if len(masks) > 10:
        print("  ...")

    mask_path = masks[mask_idx]
    print(f"\nUsando mask_idx={mask_idx} -> {mask_path.name}")

    # Cargar volumen de máscara [Z,H,W]
    vol_mask = load_volume_3d(mask_path)  # reutilizamos tu función
    Z, H, W = vol_mask.shape
    print(f"Shape máscara: Z={Z}, H={H}, W={W}")

    # ---- Info global de clases en TODO el volumen ----
    uniq, counts = np.unique(vol_mask, return_counts=True)
    print("\nClases presentes en TODO el volumen (incluyendo fondo):")
    df_global = pd.DataFrame({
        "clase": uniq.astype(int),
        "voxeles": counts,
        "porc_sobre_total": counts / counts.sum()
    })
    display(df_global)

    # En cuántas slices aparece cada clase
    print("\nNúmero de slices donde aparece cada clase:")
    rows = []
    for c in uniq:
        z_indices = np.where(vol_mask == c)[0]
        slices_unicos = np.unique(z_indices)
        rows.append({
            "clase": int(c),
            "num_slices_con_clase": int(len(slices_unicos))
        })
    df_slices = pd.DataFrame(rows).sort_values("clase")
    display(df_slices)

    # ---- Definir colormap discreto: un color por clase 0..8 ----
    # Podés cambiar estos colores si querés otra paleta:
    cmap_colors = [
        "#000000",  # 0: fondo (negro)
        "#e41a1c",  # 1: rojo
        "#377eb8",  # 2: azul
        "#4daf4a",  # 3: verde
        "#984ea3",  # 4: violeta
        "#ff7f00",  # 5: naranja
        "#ffff33",  # 6: amarillo
        "#a65628",  # 7: marrón
        "#f781bf",  # 8: rosa
    ]
    cmap = ListedColormap(cmap_colors)
    bounds = np.arange(-0.5, 9.5, 1.0)  # bordes entre clases -0.5,0.5,1.5,...,8.5
    norm = BoundaryNorm(bounds, cmap.N)

    # ---- Visualización slice a slice con slider ----
    def show_slice(z):
        slice_mask = vol_mask[z]              # [H,W]
        slice_vis  = np.rot90(slice_mask, k=k_rot)

        clases_slice, counts_slice = np.unique(slice_mask, return_counts=True)
        print(f"\nSlice z={z} -> clases presentes: {clases_slice}")
        df_slice = pd.DataFrame({
            "clase": clases_slice.astype(int),
            "voxeles": counts_slice
        })
        display(df_slice)

        plt.figure(figsize=(5, 5))
        im = plt.imshow(slice_vis, interpolation="nearest",
                        cmap=cmap, norm=norm)
        plt.title(f"Máscara Couinaud (slice {z})")
        plt.axis("off")
        cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
        cbar.set_ticks(range(9))
        cbar.set_ticklabels([str(i) for i in range(9)])  # 0..8
        plt.show()

    interact(
        show_slice,
        z=IntSlider(min=0, max=Z-1, step=1, value=Z//2)
    )

In [22]:
# Ejemplo: ver una máscara de CouinaudTr
explorar_mascara_desde_archivos(folder="train", mask_idx=0, k_rot=1)

# O una máscara de CouinaudTs
# explorar_mascara_desde_archivos(folder="test", mask_idx=3, k_rot=1)


Carpeta: /users/lvelez/PI_Velez/data/Couinaud/CouinaudTr
Total de máscaras encontradas: 161
Primeros 10 archivos de máscara (con su índice):
  [0] hepaticvessel_001.nii
  [1] hepaticvessel_002.nii
  [2] hepaticvessel_004.nii
  [3] hepaticvessel_005.nii
  [4] hepaticvessel_010.nii
  [5] hepaticvessel_011.nii
  [6] hepaticvessel_013.nii
  [7] hepaticvessel_016.nii
  [8] hepaticvessel_018.nii
  [9] hepaticvessel_020.nii
  ...

Usando mask_idx=0 -> hepaticvessel_001.nii
Shape máscara: Z=49, H=512, W=512

Clases presentes en TODO el volumen (incluyendo fondo):


,clase,voxeles,porc_sobre_total
0,0,12360207,0.962254
1,1,14104,0.001098
2,2,66908,0.005209
3,3,34647,0.002697
4,4,66292,0.005161
5,5,77487,0.006032
6,6,55188,0.004296
7,7,76491,0.005955
8,8,93732,0.007297



Número de slices donde aparece cada clase:


,clase,num_slices_con_clase
0,0,49
1,1,11
2,2,10
3,3,13
4,4,31
5,5,23
6,6,23
7,7,14
8,8,15


interactive(children=(IntSlider(value=24, description='z', max=48), Output()), _dom_classes=('widget-interact'…